# HelioSim: Basic Microgrid Simulation Demo

© 2026 The HelioSim Authors  
MIT License — see [LICENSE](../LICENSE)

This notebook demonstrates a minimal working example of the HelioSim framework:
- Generic device models (PV, battery, electrolyser, grid)
- Step-wise simulation engine
- Two control strategies: passive (green H₂ only) and active (cost-aware)

**Note**: All data is synthetic. No real client systems or proprietary logic are used.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import from heliosim core
from heliosim.core import (
    SolarIrradianceModel,
    PVPowerModel,
    GenericElectrolyser,
    BatteryStorage,
    GridInterface,
    MicrogridSystem
)

ImportError: attempted relative import with no known parent package

In [3]:
import heliosim

ModuleNotFoundError: No module named 'heliosim'

## 1. Generate Synthetic Data

In [ ]:
# Simulate 48 hours of solar irradiance (clear-sky + noise)
irradiance_model = SolarIrradianceModel(latitude=53.3, longitude=-6.2)  # Dublin-like
time_index = pd.date_range("2026-02-16", periods=48, freq="H")
ghi = irradiance_model.generate(time_index)

# PV power (kW)
pv_model = PVPowerModel(pv_capacity_kW=50)
pv_power = pv_model.power_from_irradiance(ghi)

# Load profile (synthetic residential + small industry)
np.random.seed(42)
base_load = 20 + 10 * np.sin(2 * np.pi * np.arange(48) / 24)
load = np.maximum(5, base_load + np.random.normal(0, 3, 48))

# Grid tariffs: higher at night
tariff = np.where((time_index.hour >= 7) & (time_index.hour < 20), 0.25, 0.10)  # €/kWh

## 2. Define System Topology

In [ ]:
# Devices
pv_array = {"type": "pv", "power_profile": pv_power}
battery = BatteryStorage(capacity_kWh=100, max_charge_kW=20, soc_init=0.5)
electrolyser = GenericElectrolyser(
    min_power_kW=10,
    max_power_kW=100,
    efficiency_kg_per_kWh=0.02  # generic value
)
grid = GridInterface(max_export_kW=150, max_import_kW=150)

# Assemble microgrid
microgrid = MicrogridSystem(
    devices={
        "pv": pv_array,
        "battery": battery,
        "electrolyser": electrolyser,
        "grid": grid
    },
    load_profile=load,
    tariff_profile=tariff
)

## 3. Run Simulation: Passive Mode (Green H₂ Only)

In [ ]:
results_passive = microgrid.simulate(
    strategy="passive",
    horizon_hours=48
)

## 4. Run Simulation: Active Mode (Cost-Optimized H₂)

In [ ]:
results_active = microgrid.simulate(
    strategy="active",
    horizon_hours=48,
    h2_target_kg=8.0  # produce 8 kg by end of period
)

## 5. Visualize Results

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Power flows
axs[0].plot(time_index, pv_power, label="PV Generation", color="gold")
axs[0].plot(time_index, load, label="Load", color="black", linestyle="--")
axs[0].plot(time_index, results_passive["electrolyser_power"], label="Electrolyser (Passive)", color="blue")
axs[0].plot(time_index, results_active["electrolyser_power"], label="Electrolyser (Active)", color="green")
axs[0].set_ylabel("Power (kW)")
axs[0].legend()
axs[0].grid(True)

# H₂ production
axs[1].plot(time_index, np.cumsum(results_passive["h2_produced_kg"]), label="H₂ (Passive)", color="blue")
axs[1].plot(time_index, np.cumsum(results_active["h2_produced_kg"]), label="H₂ (Active)", color="green")
axs[1].axhline(8.0, color="red", linestyle=":", label="Target (8 kg)")
axs[1].set_ylabel("Cumulative H₂ (kg)")
axs[1].set_xlabel("Time")
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()

## Conclusion

This demo shows how HelioSim enables:
- Transparent, deterministic control strategies
- Scenario comparison without black-box AI
- Reusable components for socio-technical energy modeling

All code is open-source and independent of any commercial engagement.

→ Explore more at: https://github.com/LyFX5/heliosim